# Sampling Activations from CORnet-S

Extracts layer activations from CORnet-S (DiCarlo lab) using the same optical-flow stimuli as `s01_sample_fnn.ipynb`.
CORnet-S maps to primate visual cortex areas: **V1 → V2 → V4 → IT**.

Install: `pip install git+https://github.com/dicarlolab/CORnet`

Output: `data/sampled/tensor4d_cornet_s_<AREA>_i3_n<N>_seed17.npy` (shape: neurons × 11 × 8 × 37).

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import sys, os
_d = os.path.abspath(os.getcwd())
sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))
from src.plot_utils import createFlowDataset
from time import time

from cornet import cornet_s

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

######################## PARAMS ########################
AREAS = ['V1', 'V2', 'V4', 'IT']      # CORnet-S areas in order
n_fmaps_to_sample = 40                 # max feature maps to sample per area
samples_per_fmap  = 50                 # max neurons sampled per feature map
seed              = 17
N_INSTANCES       = 3
trial_len         = 37                 # frames per stimulus trial
NDIRS             = 8
scl_factor        = 0.7
BATCH_SIZE        = 32                 # frames processed per forward pass
CORNET_SIZE       = 224               # CORnet-S input resolution

# Optical-flow stimulus parameters (must match nb01)
topdir      = '../stimuli/flowstims'
orig_shape  = (800, 600)              # PIL (width, height) of source PNGs
input_shape = (144, 256)             # (height, width) cropped for FNN
mydirs      = ['0', '45', '90', '135', '180', '225', '270', '315']
categories  = [
    'grat_W12', 'grat_W1', 'grat_W2',
    'neg1dotflow_D1_bg', 'neg1dotflow_D2_bg',
    'neg3dotflow_D1_bg', 'neg3dotflow_D2_bg',
    'pos1dotflow_D1_bg', 'pos1dotflow_D2_bg',
    'pos3dotflow_D1_bg', 'pos3dotflow_D2_bg',
]
NSTIMS = len(categories)
assert NSTIMS == 11

n_orig_imgs    = NSTIMS * NDIRS         # 88
n_shifted_imgs = n_orig_imgs * trial_len  # 3256

# ImageNet normalisation constants (CORnet-S pretrained on ImageNet)
IMGNET_MEAN = np.array([0.485, 0.456, 0.406], dtype='float32')
IMGNET_STD  = np.array([0.229, 0.224, 0.225], dtype='float32')

print(f'n_orig_imgs={n_orig_imgs}, n_shifted_imgs={n_shifted_imgs}')

Using device: cpu
n_orig_imgs=88, n_shifted_imgs=3256


In [3]:
########## LOAD MODEL + REGISTER HOOKS ##########

model = cornet_s(pretrained=True, map_location=device)
model.eval()
model = model.to(device)

activation_outputs = {}

def make_hook(area_name):
    def hook(module, input, output):
        activation_outputs[area_name] = output.detach().cpu()
    return hook

for area in AREAS:
    getattr(model.module, area).register_forward_hook(make_hook(area))

# Warm-up pass to determine output spatial shapes per area
dummy = torch.zeros(1, 3, CORNET_SIZE, CORNET_SIZE, device=device)
with torch.no_grad():
    _ = model(dummy)

area_shapes = {}  # area -> (C, H, W)
for area in AREAS:
    _, C, H, W = activation_outputs[area].shape
    area_shapes[area] = (C, H, W)
    print(f'  {area}: C={C}, H={H}, W={W}  (n_units={C*H*W})')

  V1: C=64, H=56, W=56  (n_units=200704)
  V2: C=128, H=28, W=28  (n_units=100352)
  V4: C=256, H=14, W=14  (n_units=50176)
  IT: C=512, H=7, W=7  (n_units=25088)


In [4]:
########## LOAD OPTICAL-FLOW STIMULI ##########

flow_datasets = createFlowDataset(
    categories, topdir, mydirs,
    orig_shape=orig_shape, input_shape=input_shape,
    scl_factor=scl_factor, N_INSTANCES=N_INSTANCES,
    trial_len=trial_len, stride=1,
)

for insti, arr in flow_datasets.items():
    print(f'Instance {insti}: shape={arr.shape}')  # expect (n_shifted_imgs, H*W)
assert flow_datasets[0].shape[0] == n_shifted_imgs

*INSTANCE 0 ...........
*INSTANCE 1 ...........
*INSTANCE 2 ...........
Instance 0: shape=(3256, 36864)
Instance 1: shape=(3256, 36864)
Instance 2: shape=(3256, 36864)


In [5]:
########## PREPROCESSING HELPER ##########

def preprocess_frames(flat_batch, orig_h, orig_w, target_size):
    """Ravel grayscale uint8 frames → normalised RGB tensor for CORnet-S.

    Args:
        flat_batch : (B, H*W) uint8 numpy array
        orig_h, orig_w : spatial dimensions of each raveled frame
        target_size : square side length for CORnet-S input

    Returns:
        torch.Tensor of shape (B, 3, target_size, target_size)
    """
    B = flat_batch.shape[0]
    frames = flat_batch.reshape(B, orig_h, orig_w).astype('float32') / 255.0  # (B, H, W)
    frames_rgb = np.stack([frames, frames, frames], axis=1)                    # (B, 3, H, W)
    frames_norm = (frames_rgb - IMGNET_MEAN[:, None, None]) / IMGNET_STD[:, None, None]
    t = torch.tensor(frames_norm)
    t = F.interpolate(t, size=(target_size, target_size), mode='bilinear', align_corners=False)
    return t

In [6]:
########## FORWARD PASSES (all areas in parallel) ##########

orig_h, orig_w = input_shape  # 144, 256

# Accumulate activations across instances; shape per area: (n_shifted_imgs, C, H, W)
area_outputs = {a: np.zeros((n_shifted_imgs, *area_shapes[a]), dtype='float32') for a in AREAS}

for insti in range(N_INSTANCES):
    extX = flow_datasets[insti]  # (n_shifted_imgs, H*W)
    print(f'Instance {insti}', flush=True)
    t0 = time()

    for start in range(0, n_shifted_imgs, BATCH_SIZE):
        batch_flat = extX[start:start + BATCH_SIZE]
        batch_t = preprocess_frames(batch_flat, orig_h, orig_w, CORNET_SIZE).to(device)

        with torch.no_grad():
            _ = model(batch_t)

        bs = batch_t.shape[0]
        for area in AREAS:
            area_outputs[area][start:start + bs] += activation_outputs[area].numpy()

    print(f'  done in {time()-t0:.1f}s', flush=True)

for area in AREAS:
    area_outputs[area] /= N_INSTANCES
    print(f'{area} output: {area_outputs[area].shape}  min={area_outputs[area].min():.3f}  max={area_outputs[area].max():.3f}')

Instance 0
  done in 581.8s
Instance 1
  done in 664.5s
Instance 2
  done in 726.9s
V1 output: (3256, 64, 56, 56)  min=0.000  max=2.191
V2 output: (3256, 128, 28, 28)  min=0.000  max=4.663
V4 output: (3256, 256, 14, 14)  min=0.000  max=5.642
IT output: (3256, 512, 7, 7)  min=0.000  max=19.535


In [7]:
########## SAMPLE NEURONS + BUILD TENSOR4D + SAVE (per area) ##########

os.makedirs('../data/sampled', exist_ok=True)

for area in AREAS:
    print(f'\n=== {area} ===')
    C, H, W = area_shapes[area]
    lo = area_outputs[area]           # (n_shifted_imgs, C, H, W)
    n_neurons_per_fmap = H * W

    # Reshape to (n_orig_imgs, C, H*W, trial_len)
    # lo: (n_orig_imgs*trial_len, C, H, W)
    lo_t = lo.reshape(n_orig_imgs, trial_len, C, H, W)
    lo_t = lo_t.transpose(0, 2, 3, 4, 1)              # (n_orig_imgs, C, H, W, trial_len)
    all_per_img_output = lo_t.reshape(n_orig_imgs, C, H * W, trial_len)  # (88, C, H*W, 37)

    # Per-fmap statistics for sampling weights
    all_neurons_maxs  = all_per_img_output.max(axis=(0, 3))   # (C, H*W)
    all_neurons_means = all_per_img_output.mean(axis=(0, 3))  # (C, H*W)

    # -- Sample feature maps (maxFr) --
    np.random.seed(seed)
    nfmaps = C
    maxsmean = all_neurons_maxs.mean(1)                        # (C,)
    nonzero_fmaps = int((~np.isclose(maxsmean, 0)).sum())
    n_fmaps_ = min(n_fmaps_to_sample, nonzero_fmaps)
    probs_fmap = maxsmean / maxsmean.sum()
    top_fmaps = np.random.choice(nfmaps, n_fmaps_, replace=False, p=probs_fmap)

    # -- Sample neurons within each selected fmap (maxNr) --
    samps_per_fmap = min(samples_per_fmap, n_neurons_per_fmap)
    sampled_neurons = []
    for fi in top_fmaps:
        neuron_vals = all_neurons_maxs[fi]                     # (H*W,)
        nonzero_n   = int((~np.isclose(neuron_vals, 0)).sum())
        samps       = min(samps_per_fmap, nonzero_n)
        probs_n     = neuron_vals / neuron_vals.sum()
        top_nis     = np.random.choice(n_neurons_per_fmap, samps, replace=False, p=probs_n)
        sampled_neurons += list(fi * n_neurons_per_fmap + top_nis)
    sampled_neurons = np.array(sampled_neurons)
    n_neurons_to_pick = len(sampled_neurons)
    print(f'  Sampled {n_neurons_to_pick} neurons from {n_fmaps_} feature maps')

    # -- Build tensor4d --
    tensorX     = np.zeros((n_neurons_to_pick, NSTIMS, NDIRS, trial_len), dtype='float32')
    neurons_used = np.empty((n_neurons_to_pick, 3), dtype='int')  # (fmap_idx, spatial_i, spatial_j)

    for nii, ni in enumerate(sampled_neurons):
        fi   = ni // n_neurons_per_fmap
        posi = ni %  n_neurons_per_fmap
        ii   = posi // W
        jj   = posi %  W
        neurons_used[nii] = [fi, ii, jj]
        for cati in range(NSTIMS):
            # all_per_img_output[img_idx, fmap, neuron_pos, time]
            # images are ordered: cat0_dir0, cat0_dir1, ..., cat0_dir7, cat1_dir0, ...
            pst = all_per_img_output[cati * NDIRS:(cati + 1) * NDIRS, fi, posi, :]
            tensorX[nii, cati] = pst

    print(f'  tensorX shape: {tensorX.shape}')

    # -- Save --
    SUFFIX = f'cornet_s_{area}_i{N_INSTANCES}_n{n_neurons_to_pick}_seed{seed}'
    out_tensor = f'../data/sampled/tensor4d_{SUFFIX}.npy'
    out_neurons = f'../data/sampled/neurons_used_{SUFFIX}.npy'

    if os.path.exists(out_tensor):
        print(f'  [SKIP] {out_tensor} already exists — delete to regenerate')
    else:
        np.save(out_tensor, tensorX)
        np.save(out_neurons, neurons_used)
        print(f'  Saved {out_tensor}')
        print(f'  Saved {out_neurons}')


=== V1 ===
  Sampled 2000 neurons from 40 feature maps
  tensorX shape: (2000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_cornet_s_V1_i3_n2000_seed17.npy
  Saved ../data/sampled/neurons_used_cornet_s_V1_i3_n2000_seed17.npy

=== V2 ===
  Sampled 2000 neurons from 40 feature maps
  tensorX shape: (2000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_cornet_s_V2_i3_n2000_seed17.npy
  Saved ../data/sampled/neurons_used_cornet_s_V2_i3_n2000_seed17.npy

=== V4 ===
  Sampled 2000 neurons from 40 feature maps
  tensorX shape: (2000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_cornet_s_V4_i3_n2000_seed17.npy
  Saved ../data/sampled/neurons_used_cornet_s_V4_i3_n2000_seed17.npy

=== IT ===
  Sampled 1960 neurons from 40 feature maps
  tensorX shape: (1960, 11, 8, 37)
  Saved ../data/sampled/tensor4d_cornet_s_IT_i3_n1960_seed17.npy
  Saved ../data/sampled/neurons_used_cornet_s_IT_i3_n1960_seed17.npy
